# 🤖 STAGE 3: AI Simplification Agents
## Multi-Agent System for Research Paper Simplification

**Goal:** Transform complex research into simple, accessible explanations

**What we'll build:**
1. Paper Understanding Agent (PhD-level comprehension)
2. Simplification Agent (Complex → Simple)
3. Math Explainer Agent (Equations → Plain English)
4. Critic Agent (Quality control)
5. Citation Agent (Source tracking)

**Output:** Complete simplified paper with TL;DR, explanations, and citations

---

## 📦 Step 1: Imports and Setup

In [2]:
# Core imports
import os
import json
import re
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime

# CrewAI
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI

# Vector store (from Stage 2)
from langchain.embeddings import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Environment
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("❌ OPENAI_API_KEY required for Stage 3")

print("✅ All imports successful!")
print("✅ OpenAI API Key loaded")

d:\full_end_to_end_project_implementation\Research_Paper_Simplifier\Research_Paper_Simplifier_AI\.res\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful!
✅ OpenAI API Key loaded


## 📂 Step 2: Load Stage 1 & 2 Outputs

In [3]:
# Load Stage 1 output (sections)
import json
STAGE1_OUTPUT = "data/jsons/stage1_output.json"
STAGE2_OUTPUT = "data/jsons/stage2_enhanced_output.json"
VECTORSTORE_PATH = "vectorstore_multimodal"

# Load Stage 1
with open(STAGE1_OUTPUT, 'r', encoding='utf-8') as f:
    stage1_data = json.load(f)

# Load Stage 2
with open(STAGE2_OUTPUT, 'r', encoding='utf-8') as f:
    stage2_data = json.load(f)

print("✅ Stage 1 & 2 data loaded")
print(f"\n📄 Paper: {stage1_data['metadata']['title'][:60]}...")
print(f"📊 Total searchable items: {stage2_data['extraction_stats']['total_documents']}")

✅ Stage 1 & 2 data loaded

📄 Paper: Generative Artificial Intelligence in Architecture, Engineer...
📊 Total searchable items: 93


## 🔍 Step 3: Load Vector Store

In [4]:
# Load embeddings (same as used in Stage 2)
USE_OPENAI = True

if USE_OPENAI:
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
else:
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Load vector store
vectorstore = FAISS.load_local(
    VECTORSTORE_PATH, 
    embeddings,
    allow_dangerous_deserialization=True
)

print("✅ Vector store loaded")
print("   Ready for semantic search!")

C:\Users\mdshe\AppData\Local\Temp\ipykernel_25664\900536747.py:5: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


✅ Vector store loaded
   Ready for semantic search!


## 🧠 Step 4: Initialize LLM

In [5]:
# Initialize language model
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Fast and affordable
    temperature=0.3,  # Some creativity, but mostly consistent
)

print("✅ LLM initialized: gpt-4o-mini")

✅ LLM initialized: gpt-4o-mini


## 👨‍🔬 Step 5: Create Agent 1 - Paper Understanding Agent

In [6]:
understanding_agent = Agent(
    role="Research Paper Understanding Expert",
    goal="""Read and deeply understand academic research papers at a PhD level.
    Extract the core problem, methodology, contributions, findings, and limitations.""",
    backstory="""You are a seasoned researcher with 20+ years of experience reading 
    academic papers across all disciplines. You have an exceptional ability to quickly 
    identify the core problem a paper solves, understand complex methodologies, and 
    extract key contributions. You can read between the lines and understand what 
    makes research significant.""",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Agent 1 created: Paper Understanding Expert")

✅ Agent 1 created: Paper Understanding Expert


## ✍️ Step 6: Create Agent 2 - Simplification Agent

In [7]:
simplification_agent = Agent(
    role="Science Communication Specialist",
    goal="""Transform complex academic writing into simple, accessible explanations 
    that an 8th grader can understand. Use analogies, examples, and everyday language.""",
    backstory="""You are a gifted science communicator who has spent 15 years explaining 
    complex topics to the general public. You wrote for Scientific American and hosted 
    a popular science YouTube channel. You believe that any concept, no matter how 
    complex, can be explained simply without losing accuracy. You love using analogies, 
    real-world examples, and 'Think of it like...' comparisons.""",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Agent 2 created: Simplification Specialist")

✅ Agent 2 created: Simplification Specialist


## 📐 Step 7: Create Agent 3 - Math Explainer Agent

In [8]:
math_explainer_agent = Agent(
    role="Mathematics Interpreter",
    goal="""Explain mathematical equations and formulas in plain English. 
    Break down what each part means and what the equation does in real-world terms.""",
    backstory="""You are a PhD mathematician who discovered a passion for teaching. 
    You can take any equation, no matter how complex, and explain it to a high school 
    student. You break equations into steps, explain what each variable represents, 
    and always provide real-world analogies. You make math feel intuitive rather 
    than intimidating.""",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Agent 3 created: Math Explainer")

✅ Agent 3 created: Math Explainer


## 🎯 Step 8: Create Agent 4 - Critic Agent

In [9]:
critic_agent = Agent(
    role="Quality Control Reviewer",
    goal="""Review all simplified explanations for accuracy, clarity, and completeness. 
    Identify strengths and weaknesses of both the explanations AND the original paper.""",
    backstory="""You are a tough but fair editor who has reviewed thousands of 
    scientific articles. You have a keen eye for errors, unclear explanations, and 
    missing information. You also understand the importance of balanced critique - 
    identifying both what works well and what could be improved. You check if 
    explanations are accurate, simple enough, and complete.""",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Agent 4 created: Critic & Quality Control")

✅ Agent 4 created: Critic & Quality Control


## 📎 Step 9: Create Agent 5 - Citation Agent

In [10]:
citation_agent = Agent(
    role="Citation and Source Tracker",
    goal="""Track the source of every claim and ensure all simplified statements 
    are properly linked to their original sources in the paper.""",
    backstory="""You are a meticulous research librarian with an obsessive attention 
    to detail. You ensure that every statement can be traced back to its source. 
    You note section names, page numbers, and maintain the integrity of academic 
    citation standards even in simplified explanations.""",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Agent 5 created: Citation Tracker")

✅ Agent 5 created: Citation Tracker


## 📋 Step 10: Define Tasks

In [11]:
# Get paper content
paper_abstract = stage1_data['sections_full'].get('abstract', '')
paper_introduction = stage1_data['sections_full'].get('introduction', '')
paper_title = stage1_data['metadata']['title']

# Create comprehensive paper summary for agents
paper_summary = f"""
Paper Title: {paper_title}

Abstract:
{paper_abstract[:1500]}

Introduction:
{paper_introduction[:1500]}

Available Sections: {', '.join(stage1_data['sections_full'].keys())}
"""

print("✅ Paper summary prepared for tasks")

✅ Paper summary prepared for tasks


## 📝 Step 11: Task 1 - Generate TL;DR

In [12]:
task_tldr = Task(
    description=f"""Create a TL;DR (Too Long; Didn't Read) summary of this research paper.
    
    Paper Information:
    {paper_summary}
    
    Requirements:
    1. Write 3-4 sentences maximum
    2. Capture the essence: What problem? What did they do? What did they find?
    3. Use simple, everyday language (8th grade level)
    4. Make it interesting and accessible
    
    Format:
    Start with "TL;DR:" then your summary.
    
    Example style:
    "TL;DR: This paper looks at how AI (like ChatGPT) is being used in construction. 
    The researchers reviewed 28 studies and found 7 main ways it helps, from designing 
    buildings to managing construction sites. While AI shows great promise, most 
    companies are still just experimenting rather than fully using it."
    """,
    agent=simplification_agent,
    expected_output="A 3-4 sentence TL;DR summary in simple language"
)

print("✅ Task 1 defined: Generate TL;DR")

✅ Task 1 defined: Generate TL;DR


## 📝 Step 12: Task 2 - Understand Paper Deeply

In [13]:
task_understanding = Task(
    description=f"""Analyze this research paper deeply and extract key information.
    
    Paper Information:
    {paper_summary}
    
    Extract and provide:
    1. The main problem this paper addresses
    2. The key contribution (what's new/important)
    3. The methodology used (how they did the research)
    4. Main findings (3-5 key results)
    5. Limitations mentioned
    
    Format your response as:
    
    PROBLEM:
    [One paragraph explaining the problem]
    
    CONTRIBUTION:
    [One paragraph on what's new]
    
    METHODOLOGY:
    [One paragraph on how they did it]
    
    KEY FINDINGS:
    - Finding 1
    - Finding 2
    - Finding 3
    
    LIMITATIONS:
    - Limitation 1
    - Limitation 2
    """,
    agent=understanding_agent,
    expected_output="Structured analysis with problem, contribution, methodology, findings, and limitations"
)

print("✅ Task 2 defined: Deep Understanding")

✅ Task 2 defined: Deep Understanding


## 📝 Step 13: Task 3 - Simplify Abstract

In [14]:
task_simplify_abstract = Task(
    description=f"""Simplify the abstract of this paper into plain English.
    
    Original Abstract:
    {paper_abstract}
    
    Requirements:
    1. Use 8th grade vocabulary
    2. Replace all jargon with everyday words
    3. Break complex sentences into simple ones
    4. Add analogies or examples where helpful
    5. Start with "In Simple Terms:"
    6. Keep it accurate but accessible
    
    Example transformation:
    Original: "We perform a systematic literature review using thematic analysis"
    Simplified: "We carefully read and organized 28 research papers to find common patterns"
    
    Format:
    In Simple Terms:
    [Your simplified explanation - 2-3 paragraphs]
    
    Why It Matters:
    [One sentence on why this research is important]
    """,
    agent=simplification_agent,
    expected_output="Simplified abstract in plain English with 'Why It Matters' section"
)

print("✅ Task 3 defined: Simplify Abstract")

✅ Task 3 defined: Simplify Abstract


## 📝 Step 14: Task 4 - Critical Analysis

In [15]:
task_critique = Task(
    description=f"""Review and critique this research paper objectively.
    
    Paper Information:
    {paper_summary}
    
    Provide a balanced assessment:
    
    1. STRENGTHS (What this paper does well)
       - List 3-5 strengths
       - Be specific
    
    2. WEAKNESSES (What could be improved)
       - List 3-5 limitations or weaknesses
       - Be constructive
    
    3. IMPACT (Who should care and why)
       - Who would benefit from this research?
       - What real-world applications?
    
    Format:
    
    STRENGTHS:
    ✅ [Strength 1 with brief explanation]
    ✅ [Strength 2]
    
    WEAKNESSES:
    ⚠️ [Weakness 1 with brief explanation]
    ⚠️ [Weakness 2]
    
    IMPACT:
    [Who benefits and how - 2-3 sentences]
    
    Be honest but fair. Acknowledge good work while noting areas for improvement.
    """,
    agent=critic_agent,
    expected_output="Balanced critique with strengths, weaknesses, and impact assessment"
)

print("✅ Task 4 defined: Critical Analysis")

✅ Task 4 defined: Critical Analysis


## 🚀 Step 15: Create and Execute Crew

In [16]:
# Create the crew
simplification_crew = Crew(
    agents=[
        understanding_agent,
        simplification_agent,
        critic_agent
    ],
    tasks=[
        task_tldr,
        task_understanding,
        task_simplify_abstract,
        task_critique
    ],
    process=Process.sequential,  # Tasks run one after another
    verbose=True
)

print("✅ Crew created with 3 agents and 4 tasks")
print("\n🚀 Starting simplification process...")
print("   This may take 2-3 minutes...\n")
print("="*60)

# Execute the crew
result = simplification_crew.kickoff()

print("\n" + "="*60)
print("✅ Simplification complete!")

✅ Crew created with 3 agents and 4 tasks

🚀 Starting simplification process...
   This may take 2-3 minutes...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  1b9a58e0-70d3-47da-b787-10ab0aef3a28                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a TL;DR (Too Long; Didn't Read) summary of this research paper.                                   │
│                                                                                                                 │
│      Paper Information:                                                                                         │
│                                                                                                                 │
│  Paper Title: Generative Artificial Intelligence in Architecture, Engineering, Construction, and Operations: A  │
│  Systematic Review                                                                                              │
│                                                                                                                 │
│  Abstract:                                                                                                      │
│  Generative artificial intelligence (GenAI) is a tool that can be applied to virtually all aspects              │
│  of business and life, including the construction industry. However, the adoption of GenAI                      │
│  in the construction industry, as with other innovations, is slow, and many of its applications                 │
│  thus far have been rather simplistic or failed to deliver a useful, credible output. There is                  │
│  a limited understanding of how GenAI is adopted in current practice and its potential to                       │
│  improve future practice in architecture, engineering, construction, and operations (AECO).                     │
│  Using a systematic literature review approach, this study aims to map the current issues                       │
│  in applying GenAI. The literature review initially identified 1013 peer-reviewed articles                      │
│  from ProQuest, Scopus, and Web of Science. The articles were further filtered based on                         │
│  specific criteria, resulting in 28 articles being retained for thematic analysis. The findings                 │
│  show a cluster of patterns in which GenAI is being adopted and shows promise. The                              │
│  core themes identified are as follows: (1) project brief, (2) architectural design, (3) building               │
│  information modelling, (4) structural design, (5) construction and demolition, (6) operations,                 │
│  and (7) urban governance. A typical trend noted in the AECO industry has been training                         │
│  AI models that achieve quicker results, improve quality, and use fewer resources.                              │
│  Keywords: built environment; generative artificial intelligence; construction; AECO                            │
│                                                                                                                 │
│  Introduction:                                                                                                  │
│  Construction as an industry is notorious for its slow adoption of innovation. Despite                          │
│  using computer-aided technologies for many decades in different stages of the project                          │
│  lifecycle, the industry is still slow in implementing cutting-edge technologies and remains                    │
│  traditional in its ways of working. Coupled with a considerable increase in administrative                     │
│  requirements and expectations, the productivity of the industry suffers.                                       │
│  There has been an increased interest in artificial int

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Science Communication Specialist                                                                        │
│                                                                                                                 │
│  Task: Create a TL;DR (Too Long; Didn't Read) summary of this research paper.                                   │
│                                                                                                                 │
│      Paper Information:                                                                                         │
│                                                                                                                 │
│  Paper Title: Generative Artificial Intelligence in Architecture, Engineering, Construction, and Operations: A  │
│  Systematic Review                                                                                              │
│                                                                                                                 │
│  Abstract:                                                                                                      │
│  Generative artificial intelligence (GenAI) is a tool that can be applied to virtually all aspects              │
│  of business and life, including the construction industry. However, the adoption of GenAI                      │
│  in the construction industry, as with other innovations, is slow, and many of its applications                 │
│  thus far have been rather simplistic or failed to deliver a useful, credible output. There is                  │
│  a limited understanding of how GenAI is adopted in current practice and its potential to                       │
│  improve future practice in architecture, engineering, construction, and operations (AECO).                     │
│  Using a systematic literature review approach, this study aims to map the current issues                       │
│  in applying GenAI. The literature review initially identified 1013 peer-reviewed articles                      │
│  from ProQuest, Scopus, and Web of Science. The articles were further filtered based on                         │
│  specific criteria, resulting in 28 articles being retained for thematic analysis. The findings                 │
│  show a cluster of patterns in which GenAI is being adopted and shows promise. The                              │
│  core themes identified are as follows: (1) project brief, (2) architectural design, (3) building               │
│  information modelling, (4) structural design, (5) construction and demolition, (6) operations,                 │
│  and (7) urban governance. A typical trend noted in the AECO industry has been training                         │
│  AI models that achieve quicker results, improve quality, and use fewer resources.                              │
│  Keywords: built environment; generative artificial intelligence; construction; AECO                            │
│                                                                                                                 │
│  Introduction:                                                                                                  │
│  Construction as an industry is notorious for its slow adoption of innovation. Despite                          │
│  using computer-aided technologies for many decades in different stages of the project                          │
│  lifecycle, the industry is still slow in implementing cutting-edge technologies and remains                    │
│  traditional in its ways of working. Coupled with a considerable increase in administrative                     │
│  requirements and expectations, the productivity of the

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Science Communication Specialist                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  TL;DR: This paper explores how generative AI (like smart design tools) is being used in the construction       │
│  industry. The researchers looked at 28 studies and found 7 key areas where AI can help, such as designing      │
│  buildings and managing projects. Although AI has the potential to make construction faster and better, many    │
│  companies are still just trying it out instead of fully embracing it.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quality Control Reviewer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  STRENGTHS:                                                                                                     │
│  ✅ The paper provides a comprehensive systematic literature review, synthesizing findings from 1,013 articles  │
│  down to 28 relevant studies, which adds rigor to the analysis and ensures a focused examination of GenAI       │
│  applications in the AECO sector.                                                                               │
│  ✅ The identification of seven core themes where GenAI can be applied offers valuable insights into specific   │
│  areas of potential improvement in the construction industry, making the findings actionable for                │
│  practitioners.                                                                                                 │
│  ✅ The discussion on the slow adoption of GenAI highlights a critical issue in the construction industry,      │
│  effectively framing the context for the research and underscoring the need for innovation in traditional       │
│  practices.                                                                                                     │
│  ✅ The paper emphasizes the potential benefits of GenAI, such as improved quality, efficiency, and resource    │
│  management, which can inspire stakeholders to consider integrating these technologies into their workflows.    │
│  ✅ The inclusion of various databases (ProQuest, Scopus, Web of Science) for literature sourcing enhances the  │
│  credibility and breadth of the review, ensuring a diverse range of perspectives is considered.                 │
│                                                                                                                 │
│  WEAKNESSES:                                                                                                    │
│  ⚠️ The study's reliance on only 28 articles after filtering may limit the generalizability of the findings,    │
│  as this small sample size may not capture the full spectrum of GenAI applications or the challenges faced in   │
│  the construction industry.                                                                                     │
│  ⚠️ The paper does not sufficiently address potential biases or gaps in the existing literature, which could    │
│  affect the conclusions drawn about the effectiveness and challenges of GenAI adoption, leaving readers with    │
│  an incomplete understanding of the landscape.                                                                  │
│  ⚠️ The discussion on the organizational structure of the construction industry could be expanded to provide    │
│  deeper insights into specific barriers to GenAI adoption, which would enhance the paper's practical            │
│  relevance.                                                                                                     │
│  ⚠️ The implications of the findings for future research are not clearly articulated, which could leave         │
│  readers uncertain about the next steps in exploring GenAI in the AECO sector.                                  │
│  ⚠️ The paper could benefit from a more detailed exploration of case studies or examples of successful GenAI    │
│  implementations in the construction industry to illustrate its potential impact concretely.                    │
│                                                             

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Review and critique this research paper objectively.                                                           │
│                                                                                                                 │
│      Paper Information:                                                                                         │
│                                                                                                                 │
│  Paper Title: Generative Artificial Intelligence in Architecture, Engineering, Construction, and Operations: A  │
│  Systematic Review                                                                                              │
│                                                                                                                 │
│  Abstract:                                                                                                      │
│  Generative artificial intelligence (GenAI) is a tool that can be applied to virtually all aspects              │
│  of business and life, including the construction industry. However, the adoption of GenAI                      │
│  in the construction industry, as with other innovations, is slow, and many of its applications                 │
│  thus far have been rather simplistic or failed to deliver a useful, credible output. There is                  │
│  a limited understanding of how GenAI is adopted in current practice and its potential to                       │
│  improve future practice in architecture, engineering, construction, and operations (AECO).                     │
│  Using a systematic literature review approach, this study aims to map the current issues                       │
│  in applying GenAI. The literature review initially identified 1013 peer-reviewed articles                      │
│  from ProQuest, Scopus, and Web of Science. The articles were further filtered based on                         │
│  specific criteria, resulting in 28 articles being retained for thematic analysis. The findings                 │
│  show a cluster of patterns in which GenAI is being adopted and shows promise. The                              │
│  core themes identified are as follows: (1) project brief, (2) architectural design, (3) building               │
│  information modelling, (4) structural design, (5) construction and demolition, (6) operations,                 │
│  and (7) urban governance. A typical trend noted in the AECO industry has been training                         │
│  AI models that achieve quicker results, improve quality, and use fewer resources.                              │
│  Keywords: built environment; generative artificial intelligence; construction; AECO                            │
│                                                                                                                 │
│  Introduction:                                                                                                  │
│  Construction as an industry is notorious for its slow adoption of innovation. Despite                          │
│  using computer-aided technologies for many decades in different stages of the project                          │
│  lifecycle, the industry is still slow in implementing cutting-edge technologies and remains                    │
│  traditional in its ways of working. Coupled with a considerable increase in administrative                     │
│  requirements and expectations, the productivity of the

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  1b9a58e0-70d3-47da-b787-10ab0aef3a28                                                                           │
│  Final Output: STRENGTHS:                                                                                       │
│  ✅ The paper provides a comprehensive systematic literature review, synthesizing findings from 1,013 articles  │
│  down to 28 relevant studies, which adds rigor to the analysis and ensures a focused examination of GenAI       │
│  applications in the AECO sector.                                                                               │
│  ✅ The identification of seven core themes where GenAI can be applied offers valuable insights into specific   │
│  areas of potential improvement in the construction industry, making the findings actionable for                │
│  practitioners.                                                                                                 │
│  ✅ The discussion on the slow adoption of GenAI highlights a critical issue in the construction industry,      │
│  effectively framing the context for the research and underscoring the need for innovation in traditional       │
│  practices.                                                                                                     │
│  ✅ The paper emphasizes the potential benefits of GenAI, such as improved quality, efficiency, and resource    │
│  management, which can inspire stakeholders to consider integrating these technologies into their workflows.    │
│  ✅ The inclusion of various databases (ProQuest, Scopus, Web of Science) for literature sourcing enhances the  │
│  credibility and breadth of the review, ensuring a diverse range of perspectives is considered.                 │
│                                                                                                                 │
│  WEAKNESSES:                                                                                                    │
│  ⚠️ The study's reliance on only 28 articles after filtering may limit the generalizability of the findings,    │
│  as this small sample size may not capture the full spectrum of GenAI applications or the challenges faced in   │
│  the construction industry.                                                                                     │
│  ⚠️ The paper does not sufficiently address potential biases or gaps in the existing literature, which could    │
│  affect the conclusions drawn about the effectiveness and challenges of GenAI adoption, leaving readers with    │
│  an incomplete understanding of the landscape.                                                                  │
│  ⚠️ The discussion on the organizational structure of the construction industry could be expanded to provide    │
│  deeper insights into specific barriers to GenAI adoption, which would enhance the paper's practical            │
│  relevance.                                                                                                     │
│  ⚠️ The implications of the findings for future research are not clearly articulated, which could leave         │
│  readers uncertain about the next steps in exploring GenAI in the AECO sector.                                  │
│  ⚠️ The paper could benefit from a more detailed exploratio


✅ Simplification complete!


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 📊 Step 16: Display Results

In [17]:
print("\n" + "="*60)
print("📋 SIMPLIFIED RESEARCH PAPER")
print("="*60)
print(f"\nOriginal Title: {paper_title}")
print("\n" + "="*60)
print("\n🎯 COMPLETE SIMPLIFICATION:")
print("="*60)
print(result)
print("\n" + "="*60)


📋 SIMPLIFIED RESEARCH PAPER

Original Title: Generative Artificial Intelligence in Architecture, Engineering, Construction, and Operations: A Systematic Review


🎯 COMPLETE SIMPLIFICATION:
STRENGTHS:
✅ The paper provides a comprehensive systematic literature review, synthesizing findings from 1,013 articles down to 28 relevant studies, which adds rigor to the analysis and ensures a focused examination of GenAI applications in the AECO sector.  
✅ The identification of seven core themes where GenAI can be applied offers valuable insights into specific areas of potential improvement in the construction industry, making the findings actionable for practitioners.  
✅ The discussion on the slow adoption of GenAI highlights a critical issue in the construction industry, effectively framing the context for the research and underscoring the need for innovation in traditional practices.  
✅ The paper emphasizes the potential benefits of GenAI, such as improved quality, efficiency, and resource

## 💾 Step 17: Save Output

In [18]:
# Prepare output structure
stage3_output = {
    "paper_id": stage1_data.get('pdf_path', 'unknown'),
    "processed_at": datetime.now().isoformat(),
    "original_paper": {
        "title": paper_title,
        "authors": stage1_data['metadata'].get('author', 'Unknown'),
        "pages": stage1_data['metadata'].get('pages', 0)
    },
    "simplification": {
        "full_output": str(result),
        "agents_used": [
            "Paper Understanding Expert",
            "Science Communication Specialist",
            "Quality Control Reviewer"
        ],
        "tasks_completed": [
            "TL;DR Generation",
            "Deep Understanding",
            "Abstract Simplification",
            "Critical Analysis"
        ]
    },
    "metadata": {
        "model_used": "gpt-4o-mini",
        "processing_time": "~2-3 minutes",
        "target_reading_level": "Grade 8"
    }
}

# Save to JSON
OUTPUT_FILE = "stage3_simplified_output.json"
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(stage3_output, f, indent=2, ensure_ascii=False)

print(f"✅ Output saved to: {OUTPUT_FILE}")

# Also save as readable markdown
MD_OUTPUT = "stage3_simplified_paper.md"
with open(MD_OUTPUT, 'w', encoding='utf-8') as f:
    f.write(f"# Simplified: {paper_title}\n\n")
    f.write(f"**Original Authors:** {stage1_data['metadata'].get('author', 'Unknown')}\n\n")
    f.write(f"**Simplified by AI:** {datetime.now().strftime('%Y-%m-%d')}\n\n")
    f.write("---\n\n")
    f.write(str(result))

print(f"✅ Markdown version saved to: {MD_OUTPUT}")

✅ Output saved to: stage3_simplified_output.json
✅ Markdown version saved to: stage3_simplified_paper.md


## 🔍 Step 18: RAG-Based Q&A (Bonus)

In [19]:
def ask_question_about_paper(question: str, k: int = 3):
    """Use RAG to answer questions about the paper"""
    
    # Search vector store
    relevant_docs = vectorstore.similarity_search(question, k=k)
    
    # Prepare context
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    # Create QA agent
    qa_agent = Agent(
        role="Research Paper Q&A Assistant",
        goal="Answer questions about the research paper using provided context",
        backstory="You answer questions clearly and cite your sources.",
        llm=llm,
        verbose=False
    )
    
    # Create task
    qa_task = Task(
        description=f"""Answer this question using the provided context from the paper.
        
        Question: {question}
        
        Context from paper:
        {context[:2000]}
        
        Provide a clear, simple answer. If the context doesn't contain the answer,
        say so honestly.
        """,
        agent=qa_agent,
        expected_output="Clear answer to the question"
    )
    
    # Execute
    crew = Crew(agents=[qa_agent], tasks=[qa_task], verbose=False)
    answer = crew.kickoff()
    
    return str(answer)

print("✅ Q&A function ready")
print("\nTry asking questions like:")
print('  ask_question_about_paper("What are the main themes found?")')
print('  ask_question_about_paper("What limitations were identified?")')

✅ Q&A function ready

Try asking questions like:
  ask_question_about_paper("What are the main themes found?")
  ask_question_about_paper("What limitations were identified?")


## 🧪 Step 19: Test Q&A

In [20]:
# Test with sample questions
test_questions = [
    "What are the 7 main themes identified in this paper?",
    "What are the limitations of current GenAI applications in construction?",
    "How many papers were reviewed in this study?"
]

print("🧪 Testing Q&A with sample questions:\n")

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"Q{i}: {question}")
    print(f"{'='*60}")
    answer = ask_question_about_paper(question)
    print(f"\nA{i}: {answer}")
    print()

🧪 Testing Q&A with sample questions:


Q1: What are the 7 main themes identified in this paper?

A1: The seven main themes identified in the paper are: (1) project brief, (2) architectural design, (3) structural design, (4) BIM, (5) construction and demolition, (6) operations, and (7) urban governance.


Q2: What are the limitations of current GenAI applications in construction?

A2: The context provided does not explicitly list the limitations of current GenAI applications in construction. However, it does mention that the adoption of GenAI in the construction industry is slow, and many of its applications thus far have been rather simplistic or failed to deliver a useful, credible output. Additionally, there is a limited understanding of how GenAI is adopted in current practice and its potential to improve future practice in architecture, engineering, construction, and operations (AECO). 

Therefore, while the context highlights some challenges, it does not provide a comprehensive li

## 📊 Step 20: Final Statistics

In [21]:
print("\n" + "="*60)
print("🎉 STAGE 3 COMPLETE - FINAL SUMMARY")
print("="*60)

print(f"\n📄 Original Paper:")
print(f"   Title: {paper_title[:60]}...")
print(f"   Pages: {stage1_data['metadata'].get('pages', 'Unknown')}")
print(f"   Sections: {len(stage1_data['sections_full'])}")

print(f"\n🤖 AI Processing:")
print(f"   Agents Used: 3")
print(f"   Tasks Completed: 4")
print(f"   Model: gpt-4o-mini")

print(f"\n📊 Output Generated:")
print(f"   ✅ TL;DR Summary")
print(f"   ✅ Deep Understanding Analysis")
print(f"   ✅ Simplified Abstract")
print(f"   ✅ Critical Analysis (Strengths & Weaknesses)")
print(f"   ✅ Q&A Capability (RAG-based)")

print(f"\n💾 Files Saved:")
print(f"   📄 {OUTPUT_FILE} (JSON)")
print(f"   📝 {MD_OUTPUT} (Markdown)")

print(f"\n🎯 Next Steps:")
print(f"   1. Review the simplified output")
print(f"   2. Test Q&A with your own questions")
print(f"   3. Ready for Stage 4: Build Web Interface!")

print("\n" + "="*60)
print("🎉 SUCCESS!")
print("="*60)


🎉 STAGE 3 COMPLETE - FINAL SUMMARY

📄 Original Paper:
   Title: Generative Artificial Intelligence in Architecture, Engineer...
   Pages: 19
   Sections: 4

🤖 AI Processing:
   Agents Used: 3
   Tasks Completed: 4
   Model: gpt-4o-mini

📊 Output Generated:
   ✅ TL;DR Summary
   ✅ Deep Understanding Analysis
   ✅ Simplified Abstract
   ✅ Critical Analysis (Strengths & Weaknesses)
   ✅ Q&A Capability (RAG-based)

💾 Files Saved:
   📄 stage3_simplified_output.json (JSON)
   📝 stage3_simplified_paper.md (Markdown)

🎯 Next Steps:
   1. Review the simplified output
   2. Test Q&A with your own questions
   3. Ready for Stage 4: Build Web Interface!

🎉 SUCCESS!


## 🎉 Stage 3 Complete!

**What we accomplished:**
- ✅ Created 5 specialized AI agents (used 3 in demo)
- ✅ Generated TL;DR summary
- ✅ Deep understanding analysis
- ✅ Simplified abstract in plain English
- ✅ Critical analysis (strengths & weaknesses)
- ✅ RAG-based Q&A system
- ✅ Saved outputs (JSON + Markdown)

**Cost per paper: ~$0.07**

**Ready for Stage 4: Build the Web Interface!** 🚀

---

### Try Custom Questions:

```python
# Ask your own questions
answer = ask_question_about_paper("Your question here")
print(answer)
```